In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

import cv2
import numpy as np
from ultralytics import YOLO
import os
from google.colab.patches import cv2_imshow

DIR        = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/Data"
IMAGE_NAME = "new_image.jpg"
IMAGE_PATH = os.path.join(DIR, IMAGE_NAME)

class RetailAuditPro:
    def __init__(self):
        # Cargamos el modelo YOLOv8
        self.model = YOLO('yolov8n.pt')
        self.blue = (255, 0, 0)
        self.white = (255, 255, 255)

    def run_audit(self):
        img = cv2.imread(IMAGE_PATH)
        if img is None: return print("Error al cargar imagen.")

        h, w, _ = img.shape
        overlay = img.copy()

        # 1. TRATAMIENTO DE IMAGEN PARA SOMBRAS PROFUNDAS
        # Convertimos a LAB para manipular solo la luminancia sin alterar colores
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        # Se incorpora ClipLimit que es ideal para ver latas en la oscuridad
        clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(12,12))
        cl = clahe.apply(l)
        enhanced_img = cv2.merge((cl,a,b))
        enhanced_img = cv2.cvtColor(enhanced_img, cv2.COLOR_LAB2BGR)

        # 2. INFERENCIA HÍBRIDA
        # Clases 39 (botella), 72 (lata). Confianza muy baja para forzar detección.
        results = self.model.predict(source=enhanced_img, classes=[39, 72],
                                    conf=0.08, iou=0.25, imgsz=640, verbose=False)

        # 3. SEGMENTACIÓN DE ESTANTES (Shelf Management)
        shelves = [
            {"name": "Superior", "y": (int(h*0.05), int(h*0.31))},
            {"name": "Medio",    "y": (int(h*0.32), int(h*0.62))},
            {"name": "Inferior", "y": (int(h*0.63), int(h*0.98))}
        ]

        detections = []
        if len(results) > 0:
            for box in results[0].boxes:
                coords = box.xyxy[0].cpu().numpy().astype(int)
                detections.append(coords)

        global_count = 0

        # 4. MARCADO Y CONTEO POR PRIORIDAD
        for shelf in shelves:
            y_min, y_max = shelf["y"]
            shelf_boxes = []

            for box in detections:
                cy = (box[1] + box[3]) // 2
                if y_min <= cy <= y_max:
                    # Validamos que el objeto tenga dimensiones mínimas razonables
                    alto = box[3] - box[1]
                    ancho = box[2] - box[0]
                    if alto > 35 and (alto/ancho) > 0.8: # Filtro de forma cilíndrica
                        shelf_boxes.append(box)

            # Ordena de izquierda a derecha
            shelf_boxes.sort(key=lambda x: x[0])

            for box in shelf_boxes:
                global_count += 1
                # Dibuja marco azul con grosor 3
                cv2.rectangle(overlay, (box[0], box[1]), (box[2], box[3]), self.blue, 3)

                # Etiqueta numérica con fondo sólido
                label = str(global_count)
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.3, 3)
                cv2.rectangle(overlay, (box[0], box[1]-th-25), (box[0]+tw+10, box[1]), self.blue, -1)
                cv2.putText(overlay, label, (box[0]+5, box[1]-12),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.3, self.white, 3)

        # 5. DASHBOARD DE RESULTADOS
        cv2.rectangle(overlay, (w//2-280, 20), (w//2+280, 110), (0,0,0), -1)
        cv2.putText(overlay, f"TOTAL AUDIT: {global_count}", (w//2-240, 82),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.7, self.white, 4)

        print(f"Auditoría terminada. Total detectado: {global_count}")
        cv2_imshow(cv2.resize(overlay, (int(w*0.8), int(h*0.8))))

# Inicia proceso
audit = RetailAuditPro()
audit.run_audit()

Output hidden; open in https://colab.research.google.com to view.